# Creative Pitch Pipeline (Minimal UI)

This notebook only exposes direct prototype controls and stage toggles.

- Stage order: `image_gen` -> `animation_gen` -> `image_extract` -> `upscale`
- Direct config controls: `runway_duration_seconds`, `webp_quality`, `overwrite`
- Scene controls: auto-generated checkboxes from `story.json`
- Image-gen controls: `generate_start_frame`, `generate_end_frame`, `copy_start_to_production`, `copy_end_to_production`
- Prompts are authored in `story.json` under `scene.media.generation`
- Pipeline logic lives in Python files, not notebook cells.



In [3]:
from pathlib import Path
import importlib
import sys

import ipywidgets as widgets
from IPython.display import display

start = Path.cwd().resolve()
REPO_ROOT = next((candidate for candidate in [start, *start.parents] if (candidate / "package.json").exists()), start)
PIPELINE_DIR = (REPO_ROOT / "creative-pitch" / "pipeline").resolve()
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))

import pipeline_backend
pipeline_backend = importlib.reload(pipeline_backend)
STAGES = pipeline_backend.STAGES
load_config = pipeline_backend.load_config
run_pipeline = pipeline_backend.run_pipeline

config = load_config(REPO_ROOT)
print(f"Loaded config from: {PIPELINE_DIR / 'config.json'}")
print("Stage order:", " -> ".join(stage.key for stage in STAGES))

commands = config.get("commands", {})
missing = [stage.key for stage in STAGES if not stage.default_command and not commands.get(stage.key)]
if missing:
    print("Missing commands (configure in config.json):", ", ".join(missing))


Loaded config from: /Users/ben/Documents/chaskipitch/creative-pitch/pipeline/config.json
Stage order: image_gen -> animation_gen -> image_extract -> upscale


In [4]:
defaults = config.get("defaults", {})
render = config.get("render", {})

import json

story_scene_rows = []
story_path = REPO_ROOT / "creative-pitch" / "story.json"
try:
    with story_path.open("r", encoding="utf-8") as handle:
        story_payload = json.load(handle)
    scenes_payload = story_payload.get("scenes") if isinstance(story_payload, dict) else []
    if isinstance(scenes_payload, list):
        for index, scene in enumerate(scenes_payload):
            if not isinstance(scene, dict):
                continue

            scene_id = str(scene.get("id") or f"S{index + 1}")
            title = str(scene.get("title") or scene_id)

            src_pattern = ""
            media = scene.get("media")
            if isinstance(media, dict):
                raw_pattern = media.get("srcPattern")
                if isinstance(raw_pattern, str):
                    src_pattern = raw_pattern.strip()

            active_outputs = 0
            if src_pattern.startswith("/assets/"):
                output_dir = (REPO_ROOT / "creative-pitch" / src_pattern.lstrip("/")).resolve().parent
                if output_dir.exists():
                    active_outputs = len(list(output_dir.glob("frame_*.webp")))

            story_scene_rows.append(
                {
                    "id": scene_id,
                    "title": title,
                    "active_outputs": active_outputs,
                    "default_checked": active_outputs == 0,
                }
            )
except Exception as exc:
    print(f"Warning: failed to read story scenes: {exc}")

story_scene_count = len(story_scene_rows)
default_checked_count = sum(1 for row in story_scene_rows if row["default_checked"])

runway_duration_seconds = widgets.IntSlider(
    value=max(2, min(10, int(defaults.get("runway_duration_seconds", 5)))),
    min=2,
    max=10,
    step=1,
    description="Runway s"
)

WEBP_QUALITY_PRESETS = [
    ("Low (test)", 60),
    ("Medium", 78),
    ("High (production)", 90),
]

def _nearest_webp_preset(value: int) -> int:
    options = [preset for _, preset in WEBP_QUALITY_PRESETS]
    numeric = int(value)
    return min(options, key=lambda candidate: abs(candidate - numeric))

webp_quality = widgets.Dropdown(
    options=WEBP_QUALITY_PRESETS,
    value=_nearest_webp_preset(int(render.get("webp_quality", 82))),
    description="WebP Q",
)

generate_start_frame = widgets.Checkbox(
    value=bool(defaults.get("generate_start_frame", True)),
    description="Gen Start",
    indent=False,
)
generate_end_frame = widgets.Checkbox(
    value=bool(defaults.get("generate_end_frame", False)),
    description="Gen End",
    indent=False,
)
copy_start_to_production = widgets.Checkbox(
    value=bool(defaults.get("copy_start_to_production", False)),
    description="Copy All Start Images to Production",
    indent=False,
)
copy_end_to_production = widgets.Checkbox(
    value=bool(defaults.get("copy_end_to_production", False)),
    description="Copy All End Images to Production",
    indent=False,
)

overwrite_existing = widgets.Checkbox(
    value=bool(defaults.get("overwrite", False)),
    description="Overwrite Existing Outputs",
    indent=False,
)

runway_help = widgets.HTML("<small>Runway image_to_video duration accepts integer seconds from 2 to 10.</small>")
overwrite_help = widgets.HTML("<small>If checked, selected scenes regenerate even when final WebP outputs already exist.</small>")
scene_help = widgets.HTML(
    f"<small>Loaded {story_scene_count} scene(s) from <code>creative-pitch/story.json</code>. "
    f"Default checked: {default_checked_count} scene(s) with no output frames.</small>"
)
image_help = widgets.HTML(
    "<small>Prompts are read from <code>story.json</code>. For one-pass full generation, enable both copy toggles so generated start/end keyframes are auto-promoted to production folders.</small>"
)

stage_checkboxes = {
    stage.key: widgets.Checkbox(value=True, description=stage.label, indent=False)
    for stage in STAGES
}

scene_checkboxes = {}
scene_checkbox_widgets = []
for row in story_scene_rows:
    label = f"{row['id']} - {row['title']}"
    if row["active_outputs"] > 0:
        label = f"{label} ({row['active_outputs']} existing outputs)"
    checkbox = widgets.Checkbox(
        value=bool(row["default_checked"]),
        description=label,
        indent=False,
        layout=widgets.Layout(width="100%"),
    )
    scene_checkboxes[row["id"]] = checkbox
    scene_checkbox_widgets.append(checkbox)

prototype_btn = widgets.Button(description="Apply Prototype Preset", button_style="warning")
clear_scenes_btn = widgets.Button(description="Clear All Scenes")
run_btn = widgets.Button(description="Run Selected Stages", button_style="success")
output = widgets.Output(layout=widgets.Layout(border="1px solid #ccc", max_height="500px", overflow="auto"))


def apply_prototype_preset(_):
    runway_duration_seconds.value = 2
    webp_quality.value = 60
    generate_start_frame.value = True
    generate_end_frame.value = True
    copy_start_to_production.value = True
    copy_end_to_production.value = True
    overwrite_existing.value = False


def clear_all_scenes(_):
    for checkbox in scene_checkboxes.values():
        checkbox.value = False


def logger(message: str):
    with output:
        print(message)


def on_run(_):
    output.clear_output()
    selected_stages = [key for key, checkbox in stage_checkboxes.items() if checkbox.value]
    if not selected_stages:
        with output:
            print("No stages selected.")
        return

    selected_scene_ids = [scene_id for scene_id, checkbox in scene_checkboxes.items() if checkbox.value]
    if not selected_scene_ids:
        with output:
            print("No scenes selected.")
        return

    run_config = {
        "defaults": {
            "scene_limit": 0,
            "scene_ids": selected_scene_ids,
            "runway_duration_seconds": int(runway_duration_seconds.value),
            "overwrite": bool(overwrite_existing.value),
            "generate_start_frame": bool(generate_start_frame.value),
            "generate_end_frame": bool(generate_end_frame.value),
            "copy_start_to_production": bool(copy_start_to_production.value),
            "copy_end_to_production": bool(copy_end_to_production.value),
        },
        "render": {
            "webp_quality": int(webp_quality.value)
        }
    }

    result = run_pipeline(selected_stages, config=run_config, repo_root=REPO_ROOT, logger=logger)
    with output:
        print(f"\nDone. status={result.get('status')}")


prototype_btn.on_click(apply_prototype_preset)
clear_scenes_btn.on_click(clear_all_scenes)
run_btn.on_click(on_run)

scene_selection_controls = widgets.VBox(scene_checkbox_widgets)
if scene_checkbox_widgets:
    scene_selection_controls.layout = widgets.Layout(max_height="260px", overflow="auto")
else:
    scene_selection_controls = widgets.HTML(
        "<small>No scenes with media found in <code>creative-pitch/story.json</code>.</small>"
    )

controls = widgets.VBox([
    widgets.HTML("<h3>Stage Toggles</h3>"),
    widgets.VBox(list(stage_checkboxes.values())),
    widgets.HTML("<h3>Scene Selection</h3>"),
    scene_help,
    scene_selection_controls,
    clear_scenes_btn,
    widgets.HTML("<h3>Direct Config</h3>"),
    runway_duration_seconds,
    runway_help,
    webp_quality,
    overwrite_existing,
    overwrite_help,
    widgets.HTML("<h3>Image Gen Controls</h3>"),
    widgets.HBox([generate_start_frame, generate_end_frame]),
    widgets.HBox([copy_start_to_production, copy_end_to_production]),
    image_help,
    widgets.HBox([prototype_btn, run_btn])
])

display(controls)
display(output)



Output(layout=Layout(border_bottom='1px solid #ccc', border_left='1px solid #ccc', border_right='1px solid #cc…